# 🌿 AGRI-SARTHI v2 — Production-Grade Leaf Disease Classifier
### Apple & Tomato · 4 Classes · Target: 95-99% Real-World Accuracy

---
**What changed from v1 and why:**

| Problem in v1 | Fix in v2 |
|---|---|
| MobileNetV2 frozen forever | EfficientNetV2B2 + 2-phase fine-tuning |
| Weak augmentation | Heavy real-world augmentation (blur, CLAHE-sim, cutout) |
| No class balancing | Computed class weights passed to `fit()` |
| Single softmax pass | Test-Time Augmentation (TTA) over 7 passes |
| Overconfident predictions | Label smoothing (0.1) |
| Shallow head | SE-Attention + residual dense head |
| No explainability | Grad-CAM heatmap on any image |

---

In [6]:
# ══════════════════════════════════════════════════════════════════
# CELL 1 — IMPORTS
# ══════════════════════════════════════════════════════════════════
import os, io, json, glob, shutil, warnings, math, random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as mpl_cm
import seaborn as sns
from PIL import Image, ImageEnhance, ImageFilter
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import backend as K
from tensorflow.keras.applications import EfficientNetV2B2
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import (
    GlobalAveragePooling2D, GlobalMaxPooling2D, Dense, Dropout,
    BatchNormalization, Input, Add, Multiply, Reshape, Concatenate,
    Conv2D, Activation, Lambda
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing import image as keras_image
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, Callback
)
from tensorflow.keras.losses import CategoricalCrossentropy

warnings.filterwarnings('ignore')
tf.get_logger().setLevel('ERROR')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f'TF version  : {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPU         : {gpus if gpus else "None — training on CPU (will be slow!)"}"')
if gpus:
    # Allow memory growth to avoid OOM crashes
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
print('✅ Imports done')

TF version  : 2.16.2
GPU         : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]"
✅ Imports done


In [25]:
# ══════════════════════════════════════════════════════════════════
# CELL 2 — CONFIGURATION
# ══════════════════════════════════════════════════════════════════
CONFIG = {
    # ── Paths ─────────────────────────────────────────────────────
    'source_dir'     : './dataset',
    'train_dir'      : './dataset/train',
    'val_dir'        : './dataset/val',
    'test_dir'       : './dataset/test',
    'model_path'     : 'agri_sarthi_v2_best.keras',
    'class_idx_path' : 'class_indices.json',

    # ── Classes ───────────────────────────────────────────────────
    'classes': [
        'Apple___healthy',
        'Tomato___healthy',
        'Apple___Apple_scab',
        'Tomato___Leaf_Mold',
    ],

    # ── Image / batch ─────────────────────────────────────────────
    # EfficientNetV2B2 native resolution — DO NOT change
    'img_size'   : (260, 260),
    'batch_size' : 32,

    # ── PHASE 1 : frozen base, train head only ────────────────────
    # Fast convergence of the new head before touching ImageNet weights
    'epochs_phase1'   : 20,
    'lr_phase1'       : 1e-3,
    'patience_phase1' : 6,

    # ── PHASE 2 : fine-tune entire top of base ────────────────────
    # Low LR — prevents destroying pretrained weights
    'epochs_phase2'   : 30,
    'lr_phase2'       : 1e-4,   # MUST be 10-50x lower than phase 1
    'unfreeze_layers' : 50,     # unfreeze last 80 layers of EfficientNetV2B2
    'patience_phase2' : 10,

    # ── Regularisation ────────────────────────────────────────────
    'label_smoothing' : 0.05,   # prevents overconfidence
    'dropout_rate'    : 0.40,

    # ── Prediction ───────────────────────────────────────────────
    'confidence_threshold' : 0.55,  # lower → fewer false rejections
    'tta_steps'            : 7,     # Test-Time Augmentation passes
}

CLASS_NAMES = CONFIG['classes']
NUM_CLASSES  = len(CLASS_NAMES)
IMG_SIZE     = CONFIG['img_size']

print('✅ Config loaded')
print(f'   Backbone : EfficientNetV2B2  (260×260)')
print(f'   Classes  : {NUM_CLASSES}  (Apple___healthy, Tomato___healthy, Apple___Apple_scab, Tomato___Leaf_Mold)')
print(f'   Phase 1  : {CONFIG["epochs_phase1"]} epochs  lr={CONFIG["lr_phase1"]}')
print(f'   Phase 2  : {CONFIG["epochs_phase2"]} epochs  lr={CONFIG["lr_phase2"]}  (fine-tune last {CONFIG["unfreeze_layers"]} layers)')

✅ Config loaded
   Backbone : EfficientNetV2B2  (260×260)
   Classes  : 4  (Apple___healthy, Tomato___healthy, Apple___Apple_scab, Tomato___Leaf_Mold)
   Phase 1  : 20 epochs  lr=0.001
   Phase 2  : 30 epochs  lr=0.0001  (fine-tune last 50 layers)


In [7]:
# ══════════════════════════════════════════════════════════════════
# CELL 3 — DATASET SPLIT  (70 / 15 / 15)
# ══════════════════════════════════════════════════════════════════
def prepare_dataset(cfg: dict) -> None:
    """
    Copies images from ./dataset/<class>/ into train/val/test
    sub-folders with a 70/15/15 stratified split.
    Skips any class already populated to avoid duplicating files.
    """
    for split in ['train', 'val', 'test']:
        for cls in cfg['classes']:
            os.makedirs(os.path.join(cfg[f'{split}_dir'], cls), exist_ok=True)

    print('=' * 60)
    print('  Dataset Preparation  (70 / 15 / 15 stratified split)')
    print('=' * 60)

    total = {'train': 0, 'val': 0, 'test': 0}

    for cls in cfg['classes']:
        dest_train = os.path.join(cfg['train_dir'], cls)
        if len(os.listdir(dest_train)) > 0:
            n = len(os.listdir(dest_train))
            print(f'  ↩  {cls:<48}  already split ({n} train imgs), skipping')
            continue

        src = os.path.join(cfg['source_dir'], cls)
        imgs = (
            glob.glob(f'{src}/*.[jJ][pP][gG]')
            + glob.glob(f'{src}/*.[jJ][pP][eE][gG]')
            + glob.glob(f'{src}/*.[pP][nN][gG]')
        )

        if not imgs:
            print(f'  ⚠️  No images found in: {src}')
            continue

        train_imgs, temp      = train_test_split(imgs, test_size=0.30, random_state=SEED)
        val_imgs,   test_imgs = train_test_split(temp, test_size=0.50, random_state=SEED)

        for img_list, split_key in [
            (train_imgs, 'train_dir'),
            (val_imgs,   'val_dir'),
            (test_imgs,  'test_dir'),
        ]:
            dest = os.path.join(cfg[split_key], cls)
            for f in img_list:
                shutil.copy2(f, dest)

        total['train'] += len(train_imgs)
        total['val']   += len(val_imgs)
        total['test']  += len(test_imgs)

        print(
            f'  ✓  {cls:<48}'
            f'  train={len(train_imgs):>4}  '
            f'val={len(val_imgs):>3}  '
            f'test={len(test_imgs):>3}'
        )

    print('=' * 60)
    print(f'  Totals → train={total["train"]}  val={total["val"]}  test={total["test"]}')
    print('  Dataset ready!\n')


prepare_dataset(CONFIG)

  Dataset Preparation  (70 / 15 / 15 stratified split)
  ✓  Apple___healthy                                   train=1151  val=247  test=247
  ✓  Tomato___healthy                                  train=1113  val=239  test=239
  ✓  Apple___Apple_scab                                train= 441  val= 94  test= 95
  ✓  Tomato___Leaf_Mold                                train= 666  val=143  test=143
  Totals → train=3371  val=723  test=724
  Dataset ready!



In [8]:
# ══════════════════════════════════════════════════════════════════
# CELL 4 — HEAVY AUGMENTATION + DATA GENERATORS
# ══════════════════════════════════════════════════════════════════
# WHY heavy augmentation?
# PlantVillage images are taken in controlled lab conditions.
# Real-world photos have variable lighting, angles, blur, and backgrounds.
# Augmentation bridges that gap — it's the single biggest fix for
# "good val accuracy, bad real-world accuracy".

def cutout_augmentation(img_array, num_holes=2, hole_size=30):
    """
    Randomly masks square patches with the mean colour.
    Forces the model to use distributed features instead of
    focusing on one spot (e.g., a single lesion).
    """
    h, w = img_array.shape[:2]
    result = img_array.copy()
    mean_val = img_array.mean(axis=(0, 1))
    for _ in range(num_holes):
        y = random.randint(0, h - hole_size)
        x = random.randint(0, w - hole_size)
        result[y:y+hole_size, x:x+hole_size] = mean_val
    return result


def preprocess_with_cutout(img_array):
    """Custom preprocessing: EfficientNet normalisation + random cutout."""
    # EfficientNetV2 expects pixels in [0, 255] — it applies its own normalisation internally
    # Do NOT rescale to [0,1] manually when using EfficientNetV2
    img = img_array.astype(np.float32)
    if random.random() > 0.5:
        img = cutout_augmentation(img, num_holes=2, hole_size=40)
    return tf.keras.applications.efficientnet_v2.preprocess_input(img)


def val_preprocess(img_array):
    """Val/Test preprocessing — EfficientNet normalisation only, no augmentation."""
    return tf.keras.applications.efficientnet_v2.preprocess_input(img_array.astype(np.float32))


# ── Training generator — maximum augmentation ─────────────────────
train_datagen = ImageDataGenerator(
    preprocessing_function = preprocess_with_cutout,
    rotation_range         = 40,        # leaves grow at all angles
    width_shift_range      = 0.25,
    height_shift_range     = 0.25,
    shear_range            = 0.20,
    zoom_range             = [0.75, 1.3],  # both zoom-in and zoom-out
    brightness_range       = [0.5, 1.5],   # harsh sun → deep shade
    channel_shift_range    = 30.0,          # colour cast simulation
    horizontal_flip        = True,
    vertical_flip          = True,          # overhead photos
    fill_mode              = 'reflect',     # better than 'nearest' for leaves
)

# ── Val / Test generator — clean evaluation, no augmentation ──────
eval_datagen = ImageDataGenerator(
    preprocessing_function = val_preprocess,
)

train_gen = train_datagen.flow_from_directory(
    CONFIG['train_dir'],
    target_size = IMG_SIZE,
    batch_size  = CONFIG['batch_size'],
    class_mode  = 'categorical',
    shuffle     = True,
    seed        = SEED,
)
val_gen = eval_datagen.flow_from_directory(
    CONFIG['val_dir'],
    target_size = IMG_SIZE,
    batch_size  = CONFIG['batch_size'],
    class_mode  = 'categorical',
    shuffle     = False,
)
test_gen = eval_datagen.flow_from_directory(
    CONFIG['test_dir'],
    target_size = IMG_SIZE,
    batch_size  = CONFIG['batch_size'],
    class_mode  = 'categorical',
    shuffle     = False,
)

print(f'  Training   samples : {train_gen.samples}')
print(f'  Validation samples : {val_gen.samples}')
print(f'  Test       samples : {test_gen.samples}')

# Save class index map (needed for prediction later)
with open(CONFIG['class_idx_path'], 'w') as f:
    json.dump(train_gen.class_indices, f, indent=2)
print(f'\n✅ class_indices.json saved')

# ── Reverse map: index → class name ──────────────────────────────
idx_to_class = {v: k for k, v in train_gen.class_indices.items()}
# Reorder CLASS_NAMES to match generator's alphabetical order
CLASS_NAMES_ORDERED = [idx_to_class[i] for i in range(NUM_CLASSES)]
print('Class order (generator):  ', CLASS_NAMES_ORDERED[:4], '...')

Found 3371 images belonging to 4 classes.
Found 723 images belonging to 4 classes.
Found 724 images belonging to 4 classes.
  Training   samples : 3371
  Validation samples : 723
  Test       samples : 724

✅ class_indices.json saved
Class order (generator):   ['Apple___Apple_scab', 'Apple___healthy', 'Tomato___Leaf_Mold', 'Tomato___healthy'] ...


In [9]:
# ══════════════════════════════════════════════════════════════════
# CELL 5 — CLASS WEIGHTS  (fixes class imbalance)
# ══════════════════════════════════════════════════════════════════
# PlantVillage has very unequal class sizes.
# Without this, the model ignores minority classes.

train_labels = train_gen.classes
class_weights_arr = compute_class_weight(
    class_weight = 'balanced',
    classes      = np.unique(train_labels),
    y            = train_labels,
)
class_weight_dict = {i: w for i, w in enumerate(class_weights_arr)}

print('Class weights (higher = rarer class gets more attention):')
for idx, w in class_weight_dict.items():
    print(f'  [{idx:>2}] {CLASS_NAMES_ORDERED[idx]:<48}  weight={w:.3f}')

Class weights (higher = rarer class gets more attention):
  [ 0] Apple___Apple_scab                                weight=1.911
  [ 1] Apple___healthy                                   weight=0.732
  [ 2] Tomato___Leaf_Mold                                weight=1.265
  [ 3] Tomato___healthy                                  weight=0.757


In [16]:
# ══════════════════════════════════════════════════════════════════
# CELL 6 — MODEL  (EfficientNetV2B2 + SE-Attention head)
# ══════════════════════════════════════════════════════════════════

def squeeze_excitation_block(tensor, ratio=16):
    """
    Squeeze-and-Excitation (SE) attention block.
    Recalibrates channel-wise feature responses:
    tells the model WHICH feature maps matter most for leaf diseases.
    This gives a significant accuracy boost over plain dense layers.
    """
    channels = tensor.shape[-1]
    se = Dense(channels // ratio, activation='relu')(tensor)
    se = Dense(channels,          activation='sigmoid')(se)
    return Multiply()([tensor, se])


def build_model(num_classes: int) -> Model:
    """
    Architecture:
      EfficientNetV2B2 (ImageNet, frozen for Phase 1)
      → GlobalAveragePooling  + GlobalMaxPooling (concatenated)
      → SE-Attention
      → Dense 512 → BN → Dropout
      → Dense 256 → BN → Dropout
      → Softmax output
    """
    inputs = Input(shape=(*IMG_SIZE, 3))

    # ── Backbone (frozen) ─────────────────────────────────────────
    base = EfficientNetV2B2(
        include_top   = False,
        weights       = 'imagenet',
        input_tensor  = inputs,
        include_preprocessing = False,   # we handle preprocessing ourselves
    )
    base.trainable = False  # frozen for Phase 1

    x = base.output  # shape: (None, 9, 9, 1408)

    # ── Dual pooling — captures both global structure and local peaks ─
    avg_pool = GlobalAveragePooling2D()(x)
    max_pool = GlobalMaxPooling2D()(x)
    x = Concatenate()([avg_pool, max_pool])  # (None, 2816)

    # ── SE-Attention ──────────────────────────────────────────────
    x = squeeze_excitation_block(x, ratio=16)

    # ── Dense head with residual-style dropout ────────────────────
    x = Dense(512)(x)
    x = BatchNormalization()(x)
    x = Activation('swish')(x)       # swish works better than relu for EfficientNet
    x = Dropout(CONFIG['dropout_rate'])(x)

    x = Dense(256)(x)
    x = BatchNormalization()(x)
    x = Activation('swish')(x)
    x = Dropout(CONFIG['dropout_rate'] - 0.1)(x)

    outputs = Dense(num_classes, activation='softmax', dtype='float32')(x)

    model = Model(inputs=inputs, outputs=outputs)
    return model


model = build_model(NUM_CLASSES)

print(f'  Backbone             : EfficientNetV2B2')
print(f'  Total params         : {model.count_params():,}')
trainable = sum(tf.size(w).numpy() for w in model.trainable_weights)
print(f'  Trainable (Phase 1)  : {trainable:,}')


  Backbone             : EfficientNetV2B2
  Total params         : 11,341,330
  Trainable (Phase 1)  : 2,570,420


In [13]:
for i, layer in enumerate(model.layers):
    if hasattr(layer, 'layers'):  # This means it's a sub-model
        print(f"Base model found at index {i}: {layer.name}")

In [17]:
# ══════════════════════════════════════════════════════════════════
# CELL 7 — PHASE 1 TRAINING  (frozen backbone, head only)
# ══════════════════════════════════════════════════════════════════
# Goal: get the custom head to a good starting point
# before we start touching the pretrained weights.
# Skipping this step and jumping straight to fine-tuning
# causes training instability and poor final accuracy.

model.compile(
    optimizer = tf.keras.optimizers.AdamW(
        learning_rate = CONFIG['lr_phase1'],
        weight_decay  = 1e-4,
    ),
    loss    = CategoricalCrossentropy(label_smoothing=CONFIG['label_smoothing']),
    metrics = ['accuracy'],
)

callbacks_p1 = [
    EarlyStopping(
        monitor              = 'val_accuracy',
        patience             = CONFIG['patience_phase1'],
        restore_best_weights = True,
        verbose              = 1,
    ),
    ReduceLROnPlateau(
        monitor  = 'val_loss',
        factor   = 0.5,
        patience = 3,
        min_lr   = 1e-6,
        verbose  = 1,
    ),
    ModelCheckpoint(
        filepath      = 'phase1_best.keras',
        monitor       = 'val_accuracy',
        save_best_only= True,
        verbose       = 0,
    ),
]

print('=' * 60)
print(f'  PHASE 1 — Training head only  (max {CONFIG["epochs_phase1"]} epochs)')
print('=' * 60)

history_p1 = model.fit(
    train_gen,
    epochs          = CONFIG['epochs_phase1'],
    validation_data = val_gen,
    callbacks       = callbacks_p1,
    class_weight    = class_weight_dict,
    verbose         = 1,
)

best_p1_acc = max(history_p1.history['val_accuracy'])
print(f'\n✅ Phase 1 complete — best val accuracy: {best_p1_acc:.4f}')

  PHASE 1 — Training head only  (max 20 epochs)
Epoch 1/20


2026-04-29 14:58:29.688793: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


106/106 ━━━━━━━━━━━━━━━━━━━━ 38s 303ms/step - accuracy: 0.2751 - loss: 1.6708 - val_accuracy: 0.3458 - val_loss: 1.3549 - learning_rate: 0.0010
Epoch 2/20
106/106 ━━━━━━━━━━━━━━━━━━━━ 28s 262ms/step - accuracy: 0.2709 - loss: 1.5074 - val_accuracy: 0.3555 - val_loss: 1.3524 - learning_rate: 0.0010
Epoch 3/20
106/106 ━━━━━━━━━━━━━━━━━━━━ 29s 270ms/step - accuracy: 0.2758 - loss: 1.4579 - val_accuracy: 0.3057 - val_loss: 1.3804 - learning_rate: 0.0010
Epoch 4/20
106/106 ━━━━━━━━━━━━━━━━━━━━ 30s 285ms/step - accuracy: 0.2888 - loss: 1.4331 - val_accuracy: 0.3582 - val_loss: 1.3463 - learning_rate: 0.0010
Epoch 5/20
106/106 ━━━━━━━━━━━━━━━━━━━━ 31s 294ms/step - accuracy: 0.3081 - loss: 1.4415 - val_accuracy: 0.3098 - val_loss: 1.3588 - learning_rate: 0.0010
Epoch 6/20
106/106 ━━━━━━━━━━━━━━━━━━━━ 43s 404ms/step - accuracy: 0.3142 - loss: 1.4232 - val_accuracy: 0.3485 - val_loss: 1.3544 - learning_rate: 0.0010
Epoch 7/20
106/106 ━━━━━━━━━━━━━━━━━━━━ 43s 409ms/step - accuracy: 0.2865 - loss:

In [26]:
# ══════════════════════════════════════════════════════════════════
# CELL 8 — PHASE 2 TRAINING  (fine-tune last N base layers)
# ══════════════════════════════════════════════════════════════════
# This is the KEY step that takes you from 70-80% to 95%+.
# We unfreeze the top layers of EfficientNetV2B2 so they
# adapt from "generic ImageNet features" to "leaf disease features".
#
# CRITICAL: LR must be ~50x lower than Phase 1 (2e-5 vs 1e-3).
# A high LR here will destroy the pretrained weights.

base_model = model
base_model.trainable = True

# Make all layers trainable first
for layer in model.layers:
    layer.trainable = True

# Keep BatchNorm frozen — critical for stability
for layer in model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

# Freeze early layers (everything except last 80)
backbone_layers = [l for l in model.layers 
                   if not isinstance(l, (tf.keras.layers.Dense, 
                                         tf.keras.layers.Dropout,
                                         tf.keras.layers.BatchNormalization,
                                         tf.keras.layers.Activation,
                                         tf.keras.layers.Concatenate,
                                         tf.keras.layers.Multiply,
                                         tf.keras.layers.GlobalAveragePooling2D,
                                         tf.keras.layers.GlobalMaxPooling2D))]

n_unfreeze = CONFIG['unfreeze_layers']  # 80
for layer in backbone_layers[:-n_unfreeze]:
    layer.trainable = False

trainable_p2 = sum(tf.size(w).numpy() for w in model.trainable_weights)
print(f'  Unfrozen last {n_unfreeze} base layers')
print(f'  Trainable params (Phase 2): {trainable_p2:,}')

# Recompile with much lower LR
model.compile(
    optimizer = tf.keras.optimizers.AdamW(
        learning_rate = CONFIG['lr_phase2'],
        weight_decay  = 1e-5,
    ),
    loss    = CategoricalCrossentropy(label_smoothing=CONFIG['label_smoothing']),
    metrics = ['accuracy'],
)

callbacks_p2 = [
    EarlyStopping(
        monitor              = 'val_accuracy',
        patience             = CONFIG['patience_phase2'],
        restore_best_weights = True,
        verbose              = 1,
    ),
    ReduceLROnPlateau(
        monitor  = 'val_loss',
        factor   = 0.5,
        patience = 4,
        min_lr   = 1e-7,
        verbose  = 1,
    ),
    # Save the BEST fine-tuned model
    ModelCheckpoint(
        filepath       = CONFIG['model_path'],
        monitor        = 'val_accuracy',
        save_best_only = True,
        verbose        = 1,
    ),
]

print('\n' + '=' * 60)
print(f'  PHASE 2 — Fine-tuning  (max {CONFIG["epochs_phase2"]} epochs)')
print('=' * 60)

history_p2 = model.fit(
    train_gen,
    epochs          = CONFIG['epochs_phase2'],
    validation_data = val_gen,
    callbacks       = callbacks_p2,
    class_weight    = class_weight_dict,
    verbose         = 1,
)

best_p2_acc = max(history_p2.history['val_accuracy'])
print(f'\n✅ Phase 2 complete — best val accuracy: {best_p2_acc:.4f}')
print(f'✅ Best model saved → {CONFIG["model_path"]}')

  Unfrozen last 50 base layers
  Trainable params (Phase 2): 7,492,192

  PHASE 2 — Fine-tuning  (max 30 epochs)
Epoch 1/30
106/106 ━━━━━━━━━━━━━━━━━━━━ 0s 397ms/step - accuracy: 0.3379 - loss: 1.3693
Epoch 1: val_accuracy improved from -inf to 0.45228, saving model to agri_sarthi_v2_best.keras
106/106 ━━━━━━━━━━━━━━━━━━━━ 62s 479ms/step - accuracy: 0.3380 - loss: 1.3693 - val_accuracy: 0.4523 - val_loss: 1.3218 - learning_rate: 1.0000e-04
Epoch 2/30
106/106 ━━━━━━━━━━━━━━━━━━━━ 0s 359ms/step - accuracy: 0.3564 - loss: 1.3746
Epoch 2: val_accuracy did not improve from 0.45228
106/106 ━━━━━━━━━━━━━━━━━━━━ 43s 404ms/step - accuracy: 0.3563 - loss: 1.3747 - val_accuracy: 0.3845 - val_loss: 1.3320 - learning_rate: 1.0000e-04
Epoch 3/30
106/106 ━━━━━━━━━━━━━━━━━━━━ 0s 478ms/step - accuracy: 0.3471 - loss: 1.3714
Epoch 3: val_accuracy did not improve from 0.45228
106/106 ━━━━━━━━━━━━━━━━━━━━ 57s 534ms/step - accuracy: 0.3471 - loss: 1.3714 - val_accuracy: 0.4025 - val_loss: 1.3301 - learning

KeyboardInterrupt: 

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 9 — TRAINING CURVES  (both phases on same plot)
# ══════════════════════════════════════════════════════════════════

def plot_training_history(h1, h2):
    p1_len = len(h1.history['accuracy'])
    p2_len = len(h2.history['accuracy'])
    epochs_p1 = range(1, p1_len + 1)
    epochs_p2 = range(p1_len + 1, p1_len + p2_len + 1)

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.suptitle('Agri-Sarthi v2 — Training Curves', fontsize=15, fontweight='bold')

    for ax, metric in zip(axes, ['accuracy', 'loss']):
        # Phase 1
        ax.plot(epochs_p1, h1.history[metric],
                color='royalblue', lw=2, label='Train (P1)')
        ax.plot(epochs_p1, h1.history[f'val_{metric}'],
                color='royalblue', lw=2, ls='--', label='Val (P1)')
        # Phase 2
        ax.plot(epochs_p2, h2.history[metric],
                color='darkorange', lw=2, label='Train (P2)')
        ax.plot(epochs_p2, h2.history[f'val_{metric}'],
                color='darkorange', lw=2, ls='--', label='Val (P2)')
        # Phase boundary
        ax.axvline(x=p1_len, color='red', lw=1.5, ls=':', label='Fine-tune start')
        ax.set_title(metric.capitalize(), fontsize=13)
        ax.set_xlabel('Epoch')
        ax.legend(fontsize=9)
        ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('training_curves_v2.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('📊 Saved → training_curves_v2.png')


plot_training_history(history_p1, history_p2)

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 10 — EVALUATION + CONFUSION MATRIX
# ══════════════════════════════════════════════════════════════════

best_model = load_model(CONFIG['model_path'])

print('=' * 60)
print('  EVALUATION on held-out sets')
print('=' * 60)

for name, gen in [('Validation', val_gen), ('Test', test_gen)]:
    gen.reset()
    loss, acc = best_model.evaluate(gen, verbose=0)
    print(f'  {name:<12}  Accuracy: {acc:.4f}   Loss: {loss:.4f}')

# Full report on test set
test_gen.reset()
y_pred_probs = best_model.predict(test_gen, verbose=1)
y_pred       = np.argmax(y_pred_probs, axis=1)
y_true       = test_gen.classes

print('\n  Classification Report (Test set):')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES_ORDERED))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(14, 11))
short_names = [n.replace('Tomato___', 'T.').replace('Apple___', 'A.') for n in CLASS_NAMES_ORDERED]
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Greens',
    xticklabels=short_names, yticklabels=short_names,
    linewidths=0.5,
)
plt.title('Confusion Matrix — Test Set', fontsize=13, fontweight='bold')
plt.ylabel('Actual', fontsize=11)
plt.xlabel('Predicted', fontsize=11)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig('confusion_matrix_v2.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Saved → confusion_matrix_v2.png')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 11 — TEST-TIME AUGMENTATION (TTA) PREDICTION
# ══════════════════════════════════════════════════════════════════
# TTA runs the model N times on slightly different versions of the
# same image and averages the predictions.
# This alone can give +2 to +5% accuracy on real-world images.
# It's especially effective when the model is uncertain.

_model_cache = load_model(CONFIG['model_path'])

# TTA augmentation generator — mild transforms for inference
_tta_datagen = ImageDataGenerator(
    preprocessing_function = val_preprocess,
    rotation_range         = 20,
    width_shift_range      = 0.1,
    height_shift_range     = 0.1,
    zoom_range             = 0.1,
    horizontal_flip        = True,
)


def load_and_preprocess(image_path: str) -> np.ndarray:
    """Load image, resize, convert to float32 array (no normalisation yet)."""
    img = keras_image.load_img(image_path, target_size=IMG_SIZE)
    arr = keras_image.img_to_array(img).astype(np.float32)
    return arr


def predict_with_tta(image_path: str, tta_steps: int = None) -> dict:
    """
    Predict a single image using Test-Time Augmentation.

    Args:
        image_path: Path to any leaf image (JPG, PNG, JPEG)
        tta_steps:  How many augmented passes. More = slower but more accurate.

    Returns:
        dict with keys: class, confidence, rejected, all_probs
    """
    if tta_steps is None:
        tta_steps = CONFIG['tta_steps']

    arr = load_and_preprocess(image_path)

    # Pass 0: clean prediction (no augmentation)
    clean_arr = val_preprocess(arr)
    all_preds = [_model_cache.predict(clean_arr[np.newaxis], verbose=0)[0]]

    # Passes 1..N: augmented predictions
    arr_4d = arr[np.newaxis]  # add batch dim
    for aug_arr in _tta_datagen.flow(arr_4d, batch_size=1, shuffle=False):
        all_preds.append(_model_cache.predict(aug_arr, verbose=0)[0])
        if len(all_preds) >= tta_steps + 1:
            break

    # Average probabilities across all passes
    avg_preds  = np.mean(all_preds, axis=0)
    idx        = int(np.argmax(avg_preds))
    confidence = float(avg_preds[idx])

    print(f'\n──────────────────────────────────────────────────')
    print(f'  Image      : {os.path.basename(image_path)}')
    print(f'  TTA passes : {len(all_preds)}')

    if confidence < CONFIG['confidence_threshold']:
        print(f'  ❌ Low confidence ({confidence:.2%}) — not a recognised leaf.')
        print(f'     Try a clearer, closer photo of the leaf.')
        return {'class': None, 'confidence': confidence, 'rejected': True, 'all_probs': {}}

    label = CLASS_NAMES_ORDERED[idx]
    print(f'  🌿 Predicted  : {label}')
    print(f'  📈 Confidence : {confidence:.2%}')
    print(f'──────────────────────────────────────────────────')
    print('  Full probability distribution:')
    # Show top 5 only for readability
    top5_idx = np.argsort(avg_preds)[::-1][:5]
    for i in top5_idx:
        bar = '█' * int(avg_preds[i] * 40)
        print(f'    {CLASS_NAMES_ORDERED[i]:<48} {avg_preds[i]:.2%}  {bar}')

    return {
        'class'     : label,
        'confidence': confidence,
        'rejected'  : False,
        'all_probs' : {CLASS_NAMES_ORDERED[i]: float(avg_preds[i]) for i in range(NUM_CLASSES)},
    }


print('✅ predict_with_tta() ready')
print('   Usage: predict_with_tta("./my_leaf.jpg")')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 12 — GRAD-CAM HEATMAP  (model explainability)
# ══════════════════════════════════════════════════════════════════
# Grad-CAM shows you WHAT part of the image the model focused on.
# If it's highlighting the leaf lesion → model is learning correctly.
# If it's highlighting the background → model is cheating / overfitting.
# Use this to diagnose prediction errors.

def make_gradcam_heatmap(img_array: np.ndarray, model: Model, last_conv_layer_name: str):
    """
    Generates a Grad-CAM heatmap for the predicted class.
    img_array: preprocessed single image, shape (1, H, W, 3)
    """
    grad_model = Model(
        inputs  = model.inputs,
        outputs = [model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array, training=False)
        pred_idx   = tf.argmax(predictions[0])
        class_score = predictions[:, pred_idx]

    grads       = tape.gradient(class_score, conv_outputs)
    pooled_grads= tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs= conv_outputs[0]
    heatmap     = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap     = tf.squeeze(heatmap).numpy()
    heatmap     = np.maximum(heatmap, 0) / (heatmap.max() + 1e-8)
    return heatmap


def show_gradcam(image_path: str, alpha: float = 0.45):
    """
    Display original image overlaid with Grad-CAM heatmap.
    Red areas = what the model focused on most.
    """
    # EfficientNetV2B2 last conv layer
    last_conv = 'top_activation'

    arr       = load_and_preprocess(image_path)
    input_arr = val_preprocess(arr)[np.newaxis]

    heatmap = make_gradcam_heatmap(input_arr, _model_cache, last_conv)

    # Resize heatmap to original image size
    heatmap_img = Image.fromarray(np.uint8(255 * heatmap))
    heatmap_img = heatmap_img.resize(IMG_SIZE, Image.LANCZOS)
    heatmap_np  = np.array(heatmap_img) / 255.0

    # Colourmap
    colormap  = mpl_cm.get_cmap('jet')
    cam_rgb   = colormap(heatmap_np)[:, :, :3]  # drop alpha
    orig_rgb  = arr / 255.0
    overlay   = (1 - alpha) * orig_rgb + alpha * cam_rgb
    overlay   = np.clip(overlay, 0, 1)

    # Prediction label
    preds     = _model_cache.predict(input_arr, verbose=0)[0]
    pred_idx  = int(np.argmax(preds))
    label     = CLASS_NAMES_ORDERED[pred_idx]
    conf      = preds[pred_idx]

    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    axes[0].imshow(orig_rgb)
    axes[0].set_title('Original Image', fontsize=12)
    axes[0].axis('off')
    axes[1].imshow(overlay)
    axes[1].set_title(f'Grad-CAM  →  {label}\n({conf:.2%})', fontsize=12)
    axes[1].axis('off')
    plt.suptitle('Where the model is looking', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('gradcam_output.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('📊 Saved → gradcam_output.png')


print('✅ show_gradcam() ready')
print('   Usage: show_gradcam("./my_leaf.jpg")')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 13 — JUPYTER UPLOAD WIDGET  (with TTA)
# ══════════════════════════════════════════════════════════════════
import ipywidgets as widgets
from IPython.display import display, clear_output

upload_btn = widgets.FileUpload(accept='image/*', multiple=False, description='Upload Leaf')
tta_slider = widgets.IntSlider(
    value=7, min=1, max=15, step=1,
    description='TTA passes:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='350px'),
)
show_cam_checkbox = widgets.Checkbox(value=True, description='Show Grad-CAM heatmap')
output_area = widgets.Output()

def _on_upload(change):
    with output_area:
        clear_output(wait=True)

        uploaded = upload_btn.value
        if not uploaded:
            return

        # Handle both old and new ipywidgets API
        if isinstance(uploaded, dict):
            item = list(uploaded.values())[0]
            img_bytes = item['content']
            fname = list(uploaded.keys())[0]
        else:
            item = uploaded[0]
            img_bytes = item['content']
            fname = item['name']

        # Save temp file so we can pass path to predict_with_tta
        tmp_path = f'/tmp/{fname}'
        with open(tmp_path, 'wb') as f:
            f.write(bytes(img_bytes))

        # Show the uploaded image
        display(
            Image.open(io.BytesIO(bytes(img_bytes))).convert('RGB').resize((220, 220))
        )

        # Run TTA prediction
        result = predict_with_tta(tmp_path, tta_steps=tta_slider.value)

        # Optionally show Grad-CAM
        if show_cam_checkbox.value and not result.get('rejected'):
            show_gradcam(tmp_path)


upload_btn.observe(_on_upload, names='value')

print('✅ Model loaded — upload a leaf photo below:')
display(
    widgets.VBox([
        widgets.HBox([tta_slider, show_cam_checkbox]),
        upload_btn,
        output_area,
    ])
)

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 14 — QUICK TIPS IF ACCURACY IS STILL LOW
# ══════════════════════════════════════════════════════════════════
"""
IF YOU STILL SEE POOR REAL-WORLD PREDICTIONS, try these in order:

1. CHECK YOUR PHOTOS
   - Is the leaf clearly visible and centred?
   - Is the background cluttered? (model trained on plain backgrounds)
   - Is lighting adequate? (no harsh shadows over the leaf)

2. LOWER CONFIDENCE THRESHOLD
   CONFIG['confidence_threshold'] = 0.40  # more permissive

3. INCREASE TTA STEPS
   predict_with_tta('./my_leaf.jpg', tta_steps=15)

4. RUN GRAD-CAM to check model behaviour
   show_gradcam('./my_leaf.jpg')
   → If red area is on background (not leaf) = overfitting to dataset artifacts
   → Fix: add more background-diverse images to dataset

5. ADD REAL PHOTOS TO TRAINING SET
   Even 20-30 real photos per class added to dataset/ folders,
   then re-running this notebook, will dramatically close the
   lab-to-real-world gap.

6. INCREASE unfreeze_layers to 120
   CONFIG['unfreeze_layers'] = 120  # more backbone adaptation
"""
print('📖 Tips printed above — check this cell if predictions seem off.')